# Установка необходимых провайдеров инференса для аппаратного ускорения на GPU(colab)
!pip install tensorrt onnxruntime-gpu --quiet
print("Подготовка окружения завершена.")


In [1]:

#1. Импорт библиотек и архитектура модели AntiSpoofNet

import os
import time
import logging
import urllib.request
from typing import Tuple, List, Dict, Any
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import onnxruntime as ort

# Настройка логирования для production-среды
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s] - %(message)s')
logger = logging.getLogger("AntiSpoofInference")

# Динамический выбор вычислительного устройства (CPU или GPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Целевое вычислительное устройство инициализировано: {DEVICE.type.upper()}")

class AntiSpoofNet(nn.Module):
    """
    Оптимизированная под edge-инференс архитектура Face Anti-Spoofing.
    Базовый экстрактор признаков: MobileNetV3-Small.
    """
    def __init__(self, num_classes: int = 2) -> None:
        super().__init__()
        base_model = models.mobilenet_v3_small(weights=None)
        in_features = base_model.classifier.in_features
        base_model.classifier = nn.Identity()
        
        self.features = base_model.features
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(in_features, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        # Глобальный средний пулинг (Global Average Pooling) для кастомной головы
        x = torch.mean(x.view(x.size(0), x.size(1), -1), dim=2)
        return self.classifier(x)

# Инициализация и перевод модели в режим оценки (инференса)
model = AntiSpoofNet().to(DEVICE)
model.eval()
logger.info("Топология модели успешно создана и перенесена на целевое устройство.")



2026-11-02 22:27:50,478 - [INFO] - Целевое вычислительное устройство инициализировано: CPU


AttributeError: 'Sequential' object has no attribute 'in_features'

In [ ]:
#2. Изолированный пайплайн сборки Демо-Датасета

def download_demo_assets(target_dir: str = "dataset") -> None:
    """
    Проверяет локальное хранилище и скачивает референсные примеры датасета.
    """
    subdirs = ["real", "spoof"]
    for subdir in subdirs:
        os.makedirs(os.path.join(target_dir, subdir), exist_ok=True)
        
    assets: Dict[str, str] = {
        f"{target_dir}/real/face1.jpg": "https://githubusercontent.com",
        f"{target_dir}/spoof/spoof1.jpg": "https://githubusercontent.com"
    }
    
    for path, url in assets.items():
        if os.path.exists(path):
            continue
        try:
            logger.info(f"Скачивание удаленного файла: {url} -> {path}")
            urllib.request.urlretrieve(url, path)
        except Exception as e:
            logger.error(f"Не удалось получить файл {path}: {e}. Генерация заглушки.")
            Image.new('RGB', (300, 300), color=('green' if 'real' in path else 'red')).save(path)

download_demo_assets()
logger.info("Проверка демонстрационного датасета успешно завершена.")


In [ ]:
#3. Экспорт графа в промежуточное представление ONNX

def export_to_onnx(pytorch_model: nn.Module, output_path: str = "anti_spoof_net.onnx") -> str:
    """
    Конвертирует вычислительный граф PyTorch в сериализованный формат ONNX 
    c поддержкой динамического размера батча.
    """
    # Использование более производительного инференс-режима вместо no_grad()
    with torch.inference_mode():
        dummy_tensor = torch.randn(1, 3, 224, 224, device=DEVICE)
        
        torch.onnx.export(
            pytorch_model,
            dummy_tensor,
            output_path,
            export_params=True,
            opset_version=16, # Использование современной стабильной версии опсета
            do_constant_folding=True,
            input_names=['input'],
            output_names=['output'],
            dynamic_axes={
                'input': {0: 'batch_size'},
                'output': {0: 'batch_size'}
            }
        )
    logger.info(f"Вычислительный граф ONNX успешно скомпилирован и записан в: {output_path}")
    return output_path

onnx_model_path = export_to_onnx(model)


In [ ]:
#4. Класс-клиент для инференса (AntiSpoofPredictor) с обработкой на русском языке

class AntiSpoofPredictor:
    """
    Высокоуровневая абстракция над средой выполнения ONNX Runtime.
    Автоматически оркеструет запуск между бэкендами TensorRT (FP16), CUDA и CPU.
    """
    def __init__(self, model_path: str) -> None:
        self.session = self._build_session(model_path)
        self.input_name = self.session.get_inputs().name
        self.labels: List[str] = ["REAL (Живое лицо)", "SPOOF (Атака/Фейк)"]
        
        # Пайплайн предобработки изображений (стандарт ImageNet / MobileNet)
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def _build_session(self, model_path: str) -> ort.InferenceSession:
        available_providers = ort.get_available_providers()
        
        if torch.cuda.is_available() and 'TensorRTExecutionProvider' in available_providers:
            logger.info("Обнаружена среда выполнения TensorRT. Компиляция движка TensorRT Engine (FP16)...")
            providers = [
                ('TensorRTExecutionProvider', {
                    'device_id': 0,
                    'trt_max_workspace_size': 2147483648, # Выделение 2 ГБ памяти под воркспейс
                    'trt_fp16_enable': True
                }),
                ('CUDAExecutionProvider', {'device_id': 0})
            ]
        else:
            logger.warning("Аппаратное ускорение TensorRT недоступно. Переключение на стандартное выполнение на CPU.")
            providers = ['CPUExecutionProvider']
            
        return ort.InferenceSession(model_path, providers=providers)

    def _stable_softmax(self, logits: np.ndarray) -> np.ndarray:
        """Безопасный расчет Softmax без риска переполнения (overflow) типа данных"""
        exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

    def predict(self, image_path: str) -> Tuple[str, float]:
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Целевой файл не найден: {image_path}")
            
        with Image.open(image_path).convert('RGB') as img:
            tensor_np = self.transform(img).unsqueeze(0).numpy()
            
        raw_outputs = self.session.run(None, {self.input_name: tensor_np})
        # Форма массива raw_outputs -> (1, 2)
        probabilities = self._stable_softmax(raw_outputs)
        
        class_idx = int(np.argmax(probabilities, axis=1))
        confidence = float(probabilities[0, class_idx])
        return self.labels[class_idx], confidence

# Инициализация движка инференса и прогон демонстрационных данных
predictor = AntiSpoofPredictor(onnx_model_path)

logger.info("Запуск пайплайна оценки на демонстрационных файлах...")
demo_paths = ["dataset/real/face1.jpg", "dataset/spoof/spoof1.jpg"]
for path in demo_paths:
    label, confidence = predictor.predict(path)
    print(f"[Предсказание] Идентификатор файла: {path.split('/')[-1]} -> Результат: {label} (Уверенность: {confidence * 100:.2f}%)")


In [ ]:
#5. Изолированный бенчмарк производительности (на русском языке)

def run_performance_benchmark(engine: AntiSpoofPredictor, iterations: int = 500) -> None:
    """
    Выполняет стресс-тестирование скомпилированного движка инференса.
    Включает обязательную фазу прогрева (warm-up) для исключения накладных расходов на выделение памяти.
    """
    dummy_payload = np.random.randn(1, 3, 224, 224).astype(np.float32)
    
    logger.info("Выполнение циклов прогрева оборудования (очистка накладных расходов инициализации)...")
    for _ in range(50):
        _ = engine.session.run(None, {engine.input_name: dummy_payload})
        
    logger.info(f"Запуск бенчмарка задержки в течение {iterations} непрерывных итераций...")
    start_timestamp = time.time()
    for _ in range(iterations):
        _ = engine.session.run(None, {engine.input_name: dummy_payload})
    end_timestamp = time.time()
    
    total_execution_time = end_timestamp - start_timestamp
    fps = iterations / total_execution_time
    latency_ms = (total_execution_time / iterations) * 1000
    
    print("\n" + "="*50)
    print(f" ОТЧЕТ О ПРИЗВОДИТЕЛЬНОСТИ БЕНЧМАРКА")
    print(f"Активный бэкенд выполнения: {engine.session.get_providers()}")
    print(f"Пропускная способность (FPS): {fps:.2f} кадров в секунду")
    print(f"Средняя задержка (Latency):  {latency_ms:.2f} мс")
    print("="*50 + "\n")

run_performance_benchmark(predictor)


In [ ]:

#6. Бенчмарк производительности с сохранением логов в CSV

import csv

def run_performance_benchmark(engine: AntiSpoofPredictor, iterations: int = 500, log_filename: str = "benchmark_results.csv") -> None:
    """
    Выполняет стресс-тестирование скомпилированного движка инференса.
    Включает фазу прогрева и сохраняет итоговые метрики в CSV-файл для сравнения.
    """
    dummy_payload = np.random.randn(1, 3, 224, 224).astype(np.float32)
    
    logger.info("Выполнение циклов прогрева оборудования (очистка накладных расходов инициализации)...")
    for _ in range(50):
        _ = engine.session.run(None, {engine.input_name: dummy_payload})
        
    logger.info(f"Запуск бенчмарка задержки в течение {iterations} непрерывных итераций...")
    start_timestamp = time.time()
    for _ in range(iterations):
        _ = engine.session.run(None, {engine.input_name: dummy_payload})
    end_timestamp = time.time()
    
    total_execution_time = end_timestamp - start_timestamp
    fps = iterations / total_execution_time
    latency_ms = (total_execution_time / iterations) * 1000
    
    # Форматируем имя активного провайдера для лога
    active_providers = engine.session.get_providers()
    backend_name = "TensorRT" if "TensorRTExecutionProvider" in active_providers else "CPU"
    
    # Печать отчета в консоль
    print("\n" + "="*50)
    print(f"ОТЧЕТ О ПРОИЗВОДИТЕЛЬНОСТИ БЕНЧМАРКА")
    print(f"Активный бэкенд выполнения: {active_providers}")
    print(f"Пропускная способность (FPS): {fps:.2f} кадров в секунду")
    print(f"Средняя задержка (Latency):  {latency_ms:.2f} мс")
    print("="*50 + "\n")
    
    # Проверяем, существует ли файл, чтобы записать заголовки
    file_exists = os.path.exists(log_filename)
    
    # Запись результатов в CSV-файл
    try:
        with open(log_filename, mode="a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            if not file_exists:
                # Записываем шапку таблицы, если файл создается впервые
                writer.writerow(["Временная метка", "Бэкенд", "FPS (кадры/сек)", "Задержка (мс)", "Полный список провайдеров"])
            
            current_time = time.strftime("%Y-%m-%d %H:%M:%S")
            writer.writerow([current_time, backend_name, f"{fps:.2f}", f"{latency_ms:.2f}", str(active_providers)])
            
        logger.info(f"Метрики производительности успешно сохранены в файл: {log_filename}")
    except Exception as e:
        logger.error(f"Не удалось сохранить логи в файл: {e}")

# Запуск обновленного бенчмарка
run_performance_benchmark(predictor)

